In [1]:
import torch
import torch.nn as nn
import math
from dataclasses import dataclass

@dataclass
class ModelArgs:
    n_heads: int
    dim: int
    hidden_dim: int
    dropout: float
    max_seq_len: int
    n_layers: int
    vocab_size: int

@dataclass
class FNNArgs:
    dim: int
    hidden_dim: int
    dropout: float

class MultiHeadAttention(nn.Module):
    def __init__(self, args: ModelArgs, is_causal=True):
        super().__init__()
        self.args = args
        assert args.dim % args.n_heads == 0, "dim must be divisible by n_heads"
        self.n_heads = args.n_heads
        self.head_dim = args.dim // args.n_heads
        self.is_causal = is_causal
        self.wq = nn.Linear(args.dim, args.dim, bias=False)
        self.wk = nn.Linear(args.dim, args.dim, bias=False)
        self.wv = nn.Linear(args.dim, args.dim, bias=False)
        self.wo = nn.Linear(args.dim, args.dim, bias=False)
        # 注意力的dropout
        self.attn_dropout = nn.Dropout(args.dropout)
        
        # 残差的dropout
        self.res_dropout = nn.Dropout(args.dropout)
        if self.is_causal:
            mask = torch.full((1,1,args.max_seq_len, args.max_seq_len), float('-inf'))
            mask = torch.triu(mask, diagonal=1)
            self.register_buffer('mask', mask)


    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor):
        '''
        [batch_size, seq_len, dim]
        '''
        batch_size, kv_len, dim = k.shape
        _ , q_len, _ = q.shape
        Q = self.wq(q).view(batch_size, q_len, self.n_heads, self.head_dim).transpose(1, 2)
        K = self.wk(k).view(batch_size, kv_len, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.wv(v).view(batch_size, kv_len, self.n_heads, self.head_dim).transpose(1, 2)

        atten = torch.matmul(Q, K.transpose(-2, -1)) / self.head_dim ** 0.5
        if self.is_causal : 
            atten = atten + self.mask[:, :, :q_len, :kv_len]

        score = torch.nn.functional.softmax(atten, dim=-1)

        output = self.attn_dropout(score)
        
        output = torch.matmul(score, V)

        output = output.transpose(1, 2).contiguous().view(batch_size, q_len, dim)
        
        output = self.wo(output)
        
        output = self.res_dropout(output)
        return output
        

class LayerNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        [batch_size, seq_len, dim]
        '''
        batch_size, seq_len, dim = x.shape
        x = (x - x.mean(dim=-1, keepdim=True)) / (x.std(dim=-1, keepdim=True) + self.eps)
        x = x * self.weight + self.bias
        return x

class FNN(nn.Module):
    def __init__(self, args: FNNArgs):
        super().__init__()
        self.w1 = nn.Linear(args.dim, args.hidden_dim, bias=False)
        self.w2 = nn.Linear(args.hidden_dim, args.dim, bias=False)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(args.dropout)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        [batch_size, seq_len, dim]
        '''
        x = self.w1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.w2(x)
        return x


@dataclass
class Args:
    max_seq_len: int
    dim: int
class PositionEncoding(nn.Module):
    def __init__(self, args: Args):
        super().__init__()
        self.pe = torch.zeros(args.max_seq_len, args.dim)
        self.position = torch.arange(0,args.max_seq_len).unsqueeze(1)
        '''
        PE(pos, 2i)   = sin( pos / 10000^(2i/d) )
        PE(pos, 2i+1) = cos( pos / 10000^(2i/d) )
        1 / 10000^(2i/d)
        = 10000^(-2i/d)
        = exp( ln(10000^(-2i/d)) )         ← 任何 a = exp(ln(a))
        = exp( (-2i/d) · ln(10000) )       ← log 性质:ln(a^b) = b·ln(a)
        = exp( 2i · (-ln(10000) / d) )     ← 把负号和除法挪进去
            ↑              ↑
        arange(0,d,2)   -math.log(10000.0)/d
        '''
        factor = torch.exp(torch.arange(0, args.dim, 2) * (-math.log(10000.0) / args.dim))

        # 计算 PE(pos, 2i)
        # 偶数部分
        self.pe[:, 0::2] = torch.sin(self.position * factor)
        # 奇数部分
        self.pe[:, 1::2] = torch.cos(self.position * factor)

        # 插入 batch 维度 [1, max_seq_len, dim]
        self.pe = self.pe.unsqueeze(0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.shape[1], :]
        return x


class EncoderLayer(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        self.attention = MultiHeadAttention(args, is_causal=False)
        self.attention_norm = LayerNorm(args.dim)
        self.ffn_norm = LayerNorm(args.dim)
        self.ffn = FNN(FNNArgs(dim=args.dim, hidden_dim=args.hidden_dim, dropout=args.dropout))
    
    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
        '''
        [batch_size, seq_len, dim]
        '''
        atten = self.attention(q, k, v)
        # 残差连接
        atten = atten + q
        atten = self.attention_norm(atten)
        # 这里的atten要保存下来，后面x要相加
        x = atten
        x = self.ffn(x)
        # 残差连接
        x = x + atten
        x = self.ffn_norm(x)
        return x

class Encoder(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        self.layers = nn.ModuleList([EncoderLayer(args) for _ in range(args.n_layers)])
        self.position_encoding = PositionEncoding(args)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        [batch_size, seq_len, dim]
        '''
        x = self.position_encoding(x)
        for layer in self.layers:
            x = layer(x, x, x)
        return x

# Case 1

In [2]:
args = ModelArgs(n_heads=8, dim=768, hidden_dim=768*4, dropout=0.1, max_seq_len=512, n_layers=6, vocab_size=10000)
print(args)

batch_size = 10

# 定义输入
x = torch.randn(batch_size, args.max_seq_len, args.dim)

# Encoder
encoder = Encoder(args)

encoder_output = encoder(x)

print("encoder output shape: ", encoder_output.shape)
print("encoder output: ", encoder_output)

ModelArgs(n_heads=8, dim=768, hidden_dim=3072, dropout=0.1, max_seq_len=512, n_layers=6, vocab_size=10000)
encoder output shape:  torch.Size([10, 512, 768])
encoder output:  tensor([[[-2.7675e-02, -9.6839e-01,  1.0341e-01,  ..., -1.3632e+00,
          -5.9834e-01,  1.9799e-01],
         [-1.0651e-01, -5.0748e-01,  1.9629e+00,  ..., -3.4898e-01,
          -1.6240e+00,  1.0464e+00],
         [ 1.4228e+00, -1.3144e+00,  8.0090e-01,  ...,  4.3037e-01,
          -1.6857e+00,  9.5101e-01],
         ...,
         [-2.4075e-01,  6.5152e-02,  1.2943e+00,  ..., -1.2115e+00,
          -1.5217e-01,  1.5432e+00],
         [ 6.7350e-01,  7.8679e-01,  9.3541e-01,  ..., -3.8973e-01,
          -1.5194e+00,  6.4198e-01],
         [ 1.1508e+00, -8.5475e-01, -8.2474e-05,  ...,  3.1368e-01,
           5.5713e-01,  5.8281e-01]],

        [[ 1.5378e+00, -5.0671e-01,  8.2108e-01,  ..., -1.3389e+00,
           1.8542e-01,  8.8789e-01],
         [ 1.0578e+00,  6.6291e-01, -2.1684e-01,  ...,  4.7748e-01,
       

# Decode阶段

In [3]:
class DecoderLayer(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args

        # 1. masked self-attention
        self.self_attention = MultiHeadAttention(args, is_causal=True)
        self.self_attention_norm = LayerNorm(args.dim)

        # 2. cross-attention
        self.cross_attention = MultiHeadAttention(args, is_causal=False)
        self.cross_attention_norm = LayerNorm(args.dim)

        # 3. feed forward
        self.ffn = FNN(FNNArgs(dim=args.dim, hidden_dim=args.hidden_dim, dropout=args.dropout))
        self.ffn_norm = LayerNorm(args.dim)

    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor) -> torch.Tensor:
        '''
        x: [batch_size, tgt_len, dim]
        encoder_output: [batch_size, src_len, dim]
        '''
        # masked self-attention
        h = self.self_attention(x, x, x)
        h = h + x
        h = self.self_attention_norm(h)

        # cross-attention
        c = self.cross_attention(h, encoder_output, encoder_output)
        c = c + h
        c = self.cross_attention_norm(c)

        # feed forward
        out = self.ffn(c)
        out = out + c
        out = self.ffn_norm(out)

        return out


class Decoder(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        self.position_encoding = PositionEncoding(args)
        self.layers = nn.ModuleList([DecoderLayer(args) for _ in range(args.n_layers)])

    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor) -> torch.Tensor:
        '''
        x: [batch_size, tgt_len, dim]
        encoder_output: [batch_size, src_len, dim]
        '''
        x = self.position_encoding(x)
        for layer in self.layers:
            x = layer(x, encoder_output)
        return x

In [4]:
args = ModelArgs(
    n_heads=8,
    dim=768,
    hidden_dim=768 * 4,
    dropout=0.1,
    max_seq_len=512,
    n_layers=6,
    vocab_size=10000
)

batch_size = 10
src_seq_len = 512
tgt_seq_len = 128

src_x = torch.randn(batch_size, src_seq_len, args.dim)
tgt_x = torch.randn(batch_size, tgt_seq_len, args.dim)

encoder = Encoder(args)
decoder = Decoder(args)

encoder_output = encoder(src_x)
decoder_output = decoder(tgt_x, encoder_output)

print("encoder output shape:", encoder_output.shape)
print("decoder output shape:", decoder_output.shape)

encoder output shape: torch.Size([10, 512, 768])
decoder output shape: torch.Size([10, 128, 768])


# Transformer & Loss

In [5]:
import torch.nn.functional as F
class Transformer(nn.Module):
    def __init__(self, args):
        super().__init__()
        self.args = args
        self.transformer = nn.ModuleDict(dict(
            embedding=nn.Embedding(args.vocab_size, args.dim),
            positionEncoder=PositionEncoding(args),
            layerNorm=LayerNorm(args.dim),
            dropout=nn.Dropout(args.dropout),
            encoder = Encoder(args),
            decoder = Decoder(args)
        ))

        # 输出层
        '''
        lm_head = Language Model Head(语言模型头部), 
        负责把 Decoder 输出的 d 维隐藏向量 投影成 V 维 logits(词表上每个 token 的 "未归一化分数"), 
        供后续 softmax 算概率分布、最终预测下一个 token。
        '''
        self.lm_head = nn.Linear(args.dim, args.vocab_size, bias=False)

        print("Transformer model initialized. %.2f M" % (self.get_num_params() / 1e6, ))
    '''统计所有参数的数量'''
    def get_num_params(self):
        return sum([p.numel() for p in self.parameters()])
    
    def forward(self, src_input, tgt_input, targets = None):
        batch, seq_len = src_input.size()
        assert seq_len <= self.args.max_seq_len

        # Embedding
        src_input_emb = self.transformer.embedding(src_input)
        tgt_input_emb = self.transformer.embedding(tgt_input)

        # Position Encoding
        src_input_emb = self.transformer.positionEncoder(src_input_emb)
        tgt_input_emb = self.transformer.positionEncoder(tgt_input_emb)

        # Dropout
        src_input_emb = self.transformer.dropout(src_input_emb)
        tgt_input_emb = self.transformer.dropout(tgt_input_emb)

        # Encoder
        src_output = self.transformer.encoder(src_input_emb)

        # Decoder
        tgt_output = self.transformer.decoder(tgt_input_emb, src_output)

        # targets 是训练时候的标准答案，用于计算损失
        if targets is not None:
            logits = self.lm_head(tgt_output)
            # logits: (batch_size, Q_seq_len, vocab_size) 询问了Q_seq_len个token, 每个Q_seq_len都有vocab_size个logits，每个logits表示第i个Q的第j个token的概率
            # targets: (batch_size, Q_seq_len)
            # targets是正确答案的索引，会将正确的位置置为1，其他位置置为0
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), reduction='mean')
        else:
            # tgt_output: (batch_size, Q_seq_len, d)
            # tgt_output[:, [-1], :] 取最后一个token的隐藏向量，作为下一个token的输入,也就是保留了最新预测出来的token
            logits = self.lm_head(tgt_output[:, [-1], :])
            loss = None
        return logits,loss


Transformer(args)


Transformer model initialized. 114.50 M


Transformer(
  (transformer): ModuleDict(
    (embedding): Embedding(10000, 768)
    (positionEncoder): PositionEncoding()
    (layerNorm): LayerNorm()
    (dropout): Dropout(p=0.1, inplace=False)
    (encoder): Encoder(
      (layers): ModuleList(
        (0-5): 6 x EncoderLayer(
          (attention): MultiHeadAttention(
            (wq): Linear(in_features=768, out_features=768, bias=False)
            (wk): Linear(in_features=768, out_features=768, bias=False)
            (wv): Linear(in_features=768, out_features=768, bias=False)
            (wo): Linear(in_features=768, out_features=768, bias=False)
            (attn_dropout): Dropout(p=0.1, inplace=False)
            (res_dropout): Dropout(p=0.1, inplace=False)
          )
          (attention_norm): LayerNorm()
          (ffn_norm): LayerNorm()
          (ffn): FNN(
            (w1): Linear(in_features=768, out_features=3072, bias=False)
            (w2): Linear(in_features=3072, out_features=768, bias=False)
            (act):

# Case 1

In [6]:
args = ModelArgs(n_heads=8, dim=768, hidden_dim=768*4, dropout=0.1, max_seq_len=512, n_layers=6, vocab_size=5000)
print(args)
special_vocabs = {
    "PAD": 0,
    "BOS": 1,
    "EOS": 2
}

ModelArgs(n_heads=8, dim=768, hidden_dim=3072, dropout=0.1, max_seq_len=512, n_layers=6, vocab_size=5000)


In [11]:
import torch
import numpy as np
import random
def seed_torch(seed=1):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_torch(42)

batch_size = 4

'''
src input = <BOS> <word1> <word2> <word3> <EOS>
target input = <BOS> <word1> <word2> <word3>
target output = <word1> <word2> <word3> <EOS>
'''

bos = special_vocabs["BOS"] + torch.zeros(batch_size, 1, dtype=torch.long)
eos = special_vocabs["EOS"] + torch.zeros(batch_size, 1, dtype=torch.long)
print(bos)
print(eos)

src = torch.randint(len(special_vocabs), args.vocab_size, (batch_size, args.max_seq_len - 2))
print(src)
tgt = torch.randint(len(special_vocabs), args.vocab_size, (batch_size, args.max_seq_len - 1))
print(tgt)

src_input = torch.cat([bos, src, eos], dim=1)
print(src_input)
target_input = torch.cat([tgt, eos], dim=1)
print(target_input)
target_output = torch.cat([tgt, eos], dim=1)
print(target_output)


tensor([[1],
        [1],
        [1],
        [1]])
tensor([[2],
        [2],
        [2],
        [2]])
tensor([[3305, 4975, 3320,  ..., 2495, 2866, 3889],
        [3952, 4644, 3044,  ...,  440, 2856, 2981],
        [4898, 3363, 3613,  ..., 4076, 2199, 4747],
        [2035, 3092, 1136,  ..., 3088, 4408, 1299]])
tensor([[ 556, 2724,  989,  ...,  624, 1490, 2270],
        [1802, 4560,   51,  ..., 2118, 4115, 3290],
        [2383, 2682,  346,  ..., 4520,  378, 2135],
        [1173, 2700, 3232,  ..., 4604,  657,  972]])
tensor([[   1, 3305, 4975,  ..., 2866, 3889,    2],
        [   1, 3952, 4644,  ..., 2856, 2981,    2],
        [   1, 4898, 3363,  ..., 2199, 4747,    2],
        [   1, 2035, 3092,  ..., 4408, 1299,    2]])
tensor([[ 556, 2724,  989,  ..., 1490, 2270,    2],
        [1802, 4560,   51,  ..., 4115, 3290,    2],
        [2383, 2682,  346,  ...,  378, 2135,    2],
        [1173, 2700, 3232,  ...,  657,  972,    2]])


In [14]:
# 模型

trans = Transformer(args)
print("模型结构: ", trans)

logits, loss = trans(src_input, target_input)
decode_ids = logits.argmax(dim=-1)
print("logits: ", logits.shape)
print("decode ids: ", decode_ids)

Transformer model initialized. 106.82 M
模型结构:  Transformer(
  (transformer): ModuleDict(
    (embedding): Embedding(5000, 768)
    (positionEncoder): PositionEncoding()
    (layerNorm): LayerNorm()
    (dropout): Dropout(p=0.1, inplace=False)
    (encoder): Encoder(
      (layers): ModuleList(
        (0-5): 6 x EncoderLayer(
          (attention): MultiHeadAttention(
            (wq): Linear(in_features=768, out_features=768, bias=False)
            (wk): Linear(in_features=768, out_features=768, bias=False)
            (wv): Linear(in_features=768, out_features=768, bias=False)
            (wo): Linear(in_features=768, out_features=768, bias=False)
            (attn_dropout): Dropout(p=0.1, inplace=False)
            (res_dropout): Dropout(p=0.1, inplace=False)
          )
          (attention_norm): LayerNorm()
          (ffn_norm): LayerNorm()
          (ffn): FNN(
            (w1): Linear(in_features=768, out_features=3072, bias=False)
            (w2): Linear(in_features=3072, ou

In [15]:
logits, loss = trans(src_input, target_input, target_output)
print("logits: ", logits.shape, logits)
print("loss: ", loss)